# View a remote plate.zarr in napari (streamed over HTTP)

The remote-viewing counterpart to `view_plate_in_napari.ipynb`: every cell below is
the same as that notebook, just pointed at a `plate.zarr` streamed live from an HPC
on-demand Jupyter session instead of a local/already-mounted path.

Before running this notebook:

1. In an on-demand Jupyter session on the cluster, run
   `serve_plate_for_remote_viewing.ipynb` -- it starts a background HTTP server for
   your `plate.zarr` and prints the exact `ssh` tunnel command(s) to run next.
2. Run the printed tunnel command from your own laptop (or, for the reverse-tunnel
   fallback, from an interactive terminal in the on-demand session plus one more
   command from your laptop -- see that notebook's own printed output).
3. Set `PLATE_PATH` below to the `http://localhost:<port>/plate.zarr` URL it printed.

Needs `blimp` importable alongside napari and ngio -- from the
`napari-feature-classifier` environment, run from wherever your own blimp checkout
lives:

```
pip install -e /path/to/blimp --no-deps
```

In [ ]:
import napari
from ngio import NgioFileNotFoundError, open_ome_zarr_plate, open_ome_zarr_container

from blimp.napari_utils import add_blimp_napari_methods

# The URL that serve_plate_for_remote_viewing.ipynb printed (run inside the on-demand
# session), reached through the ssh tunnel it also printed -- see that notebook.
PLATE_PATH = "http://localhost:8000/plate.zarr"

plate = open_ome_zarr_plate(store=PLATE_PATH, mode="r")
wells = plate.wells_paths()
print("Wells with data:", wells)

WELL = wells[0]  # change to e.g. "C/09" to pick a different well
print("Viewing well:", WELL)

## Whole plate at once, laid out like a physical plate

`viewer.add_plate(PLATE_PATH)` builds a lazy, full-resolution pyramid for every
populated well at its true row/column grid position and adds it to the viewer --
one `Image` layer per channel, a `Well_ROI_table` outline for every well (visible by
default), a `Labels` layer per label found on any well (hidden by default), and an
`FOV_ROI_table` outline for every field of view (hidden by default -- turn it on once
zoomed into a single well). Point-object tables are not included here -- that level of
per-object detail belongs in the per-well view below.

Each `Labels` layer also carries every contributing well's own measurements, merged
into one plate-wide `.features` table -- usable directly with a tool like
[napari-feature-visualization](https://github.com/fractal-napari-plugins-collection/napari-feature-visualization)
to heatmap-color objects by a feature across every well at once, not just one. Since
raw label IDs are only unique *within* one well, each well's own IDs get a
well-specific offset first (`blimp.ome_ngff.labels.well_label_offset`) so nothing
collides at plate scale -- this promotes the layer's data to `int64`.

This stays fast and full resolution regardless of how sparse the plate is:
`blimp.ome_ngff.plate.build_plate_pyramid` builds the canvas as a `dask.array.zeros`
placeholder (defined analytically -- dask never touches individual chunks just to
construct one) and overlays only populated wells' real data via chunk-aligned
assignment, so cost scales with how many wells actually have data, not with the
plate's declared grid size -- and, over this remote store, with how many wells are
actually fetched over the tunnel, not the whole store. A small gap (5% of tile size,
computed per pyramid level) is left between adjacent wells so touching wells stay
visually distinct.

In [ ]:
plate_viewer = add_blimp_napari_methods(napari.Viewer())
plate_viewer.add_plate(PLATE_PATH)

## Plate-wide feature heatmap

Coloring the plate-wide `Labels` layer above by a feature (e.g. with
[napari-feature-visualization](https://github.com/fractal-napari-plugins-collection/napari-feature-visualization))
can crash: that plugin builds a dense lookup array sized to the *largest* label ID
present, and our plate-wide IDs (`well_label_offset`-scaled to stay unique across
every well) can run into the trillions across a full plate -- fine for one well's own
compact local IDs, not for this.

`viewer.add_feature_heatmap(PLATE_PATH, label_name, feature_name)` sidesteps this
entirely: it builds an ordinary continuous-colormap `Image` layer where each object's
own pixels hold its own measurement directly (a plain float, `NaN` for background or
unmeasured objects) -- no per-ID colormap lookup involved at all. This is a genuinely
new layer, added alongside the `Labels` layer above (which stays exactly as it was) --
the per-well view further down is where Napari Feature Visualizer remains a good fit,
since one well's own local IDs are small.

Not sure which `feature_name` values are available? Merge every well's own features
table the same way `add_feature_heatmap` does, and look at its columns:

```python
from blimp.ome_ngff.plate import _read_plate_wide_features

features_df = _read_plate_wide_features(PLATE_PATH, "Nuclei", kind="mip")
print(features_df.columns.tolist())
```

Building the heatmap costs roughly one pass over every *populated* well's own label
pixels, not the plate's declared grid size -- but for a large, fully-populated plate
this can still take a couple of minutes if you ask for every well at once (slower
still over this remote store than a local path, since every chunk is now a network
round trip through the tunnel). Pass `wells="C/09"` (or a list of well paths) to
restrict this to only the well(s) you actually need.

In [ ]:
plate_viewer.add_feature_heatmap(PLATE_PATH, "Nuclei", "Nuclei_area")

## Per-well detail: image, labels, measurements, ROIs

Hands the well's image group to the `napari-ome-zarr` plugin reader directly.
OME-NGFF's multiscale pyramid and channel colors are core spec, not a blimp-specific
convention -- the plugin already reads both correctly and lazily (dask-backed, so
napari picks whichever pyramid level fits the current zoom instead of decoding the
full-resolution array up front). It even auto-discovers any labels as plain `Labels`
layers, since OME-NGFF labels are core spec too -- see the next cell for attaching
their measurements.

In [ ]:
# A plain PLATE_PATH / WELL / kind pathlib join (as the local-path notebook uses)
# silently mangles a URL -- Path collapses "http://" to "http:/" on any /-join -- so
# this joins as a plain string instead, and probes each candidate by actually trying
# to open it rather than Path(...).exists() (meaningless for a URL).
for kind in ("mip", "stack"):
    image_group_path = f"{PLATE_PATH}/{WELL}/{kind}"
    try:
        open_ome_zarr_container(image_group_path)
        break
    except NgioFileNotFoundError:
        continue
else:
    raise FileNotFoundError(f"No mip or stack image found for well {WELL}")

viewer = add_blimp_napari_methods(napari.Viewer())
layers = viewer.open(image_group_path, plugin="napari-ome-zarr")
print(f"Opened '{kind}':", [layer.name for layer in layers])

## Add measurements, point objects, and FOV boundaries

The cell above's `Labels` layers (if any) are bare pixel data -- napari-ome-zarr has no
idea blimp attached a linked `FeatureTable` to each one, since that's a blimp/Fractal/ngio
convention, not core OME-NGFF spec. Rather than reading the label a second time via
`blimp.napari_utils.add_labels_with_measurements`, attach each one's measurements
directly onto the layer already loaded above.

Point-object tables (spots/blobs with no stable pixel identity -- see
`blimp.ome_ngff.labels._write_well_points`) and FOV boundaries genuinely have no core-spec
equivalent at all, so those still go through blimp's own helpers. A well with none of
these just adds nothing here.

In [ ]:
container = open_ome_zarr_container(image_group_path)

for layer in layers:
    if isinstance(layer, napari.layers.Labels):
        label_name = layer.name.rsplit("/", 1)[-1]  # napari-ome-zarr nests it under "labels/labels/<name>"
        table_name = f"{label_name}_features"
        if table_name in container.list_tables():
            layer.features = container.get_feature_table(table_name).dataframe.reset_index()
            print(f"Attached {len(layer.features)} rows of measurements to '{layer.name}'")

for table_name in container.list_tables():
    if container.get_table(table_name).table_type() == "generic_roi_table":
        viewer.add_points_with_measurements(image_group_path, table_name)

if "FOV_ROI_table" in container.list_tables():
    viewer.add_rois(image_group_path)